# XTTS-v2 fine-tune &mdash; VoiceMakers Sinhala female voices

Dinithi (4.82 h) + Harini (2.14 h) &asymp; **7 h across two distinct female speakers**.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** or **GPU P100** — either works; the scripts pin to one GPU |
| Internet | **ON** |
| Input | `SinhalaTTS_Dataset_Publication_by_VoiceMakers` (already attached) |
| Persistence | **Files** &mdash; needed to resume past the 12 h session limit |

## Two things this run does differently

**The text becomes ASCII before it reaches the tokenizer.** XTTS-v2's `vocab.json` is a
whitespace-pretokenised BPE with an `[UNK]` fallback and contains no Sinhala codepoint &mdash;
nor the diacritics this corpus romanises with (`ā ī ū ē ṭ ḍ ṇ ḷ ṁ` are all missing). Fed
either column raw, **every word becomes one `[UNK]`** and the model trains on "unknown
unknown unknown": loss falls, audio is noise. Cell 4 asserts 0 `[UNK]` before any GPU time
is spent.

**Two speakers, correctly labelled.** XTTS samples a conditioning clip from the *same
speaker* on every step. Two real labels teach "the reference predicts the voice"; pooling
them under one label teaches the opposite, and no amount of data repairs it.

## 1. Install &mdash; restart the session after this cell

In [ ]:
# coqui-tts is the maintained idiap fork. Do NOT `pip install TTS` -- that one
# pins torch<2.1 and replaces Kaggle's CUDA build with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile librosa tensorboard

import os
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "\n*** NO GPU. Settings -> Accelerator -> GPU T4 x2 or P100, then restart. ***")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
    # Save & Run All cannot restart the session, and does not need to: every step
    # that imports TTS runs in a fresh subprocess and so picks these packages up
    # regardless of what this notebook process already imported.
    print("\nBatch mode -- no restart needed, continuing straight on.")
else:
    print("\n>>> Now: Run -> Restart session, then continue from the NEXT cell. <<<")


## 2. Code and paths

In [ ]:
import os, shutil, subprocess, pathlib

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
CODE = "/kaggle/working/Dataset-creation-withEmotion"

# Always clone fresh rather than pulling. With Persistence -> Files on, a stale
# checkout survives between sessions, and `git pull --ff-only` cannot fast-forward
# across a rewritten history -- it fails, and a swallowed failure would leave you
# running old code while believing it was current. A shallow clone costs seconds.
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
SRC = CODE + "/xtts_model_female"
print("code at", subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# Locate the attached dataset by finding the speaker folder and taking its
# parent. Kaggle nests inputs differently depending on how they were attached,
# so anchoring on a known folder name beats guessing the mount path.
inp = pathlib.Path("/kaggle/input")
hits = [p for p in inp.rglob("*") if p.is_dir() and "dinithi" in p.name.lower()] \
       if inp.is_dir() else []
DATA = str(min(hits, key=lambda p: len(p.parts)).parent) if hits else None

print("code   :", SRC)
print("data   :", DATA)
if DATA is None:
    raise RuntimeError(
        "Dataset not found. Attach 'SinhalaTTS Dataset Publication by VoiceMakers' "
        "via Add Input. Directories seen under /kaggle/input: "
        + str([p.name for p in inp.iterdir()] if inp.is_dir() else []))

# /kaggle/working is a 20 GB volume and a GPTTrainer checkpoint is ~5.5 GB --
# model plus AdamW state. The trainer holds up to four at once: best_model_<step>
# .pth, the full copy of it that trainer.io.save_best_model writes out as
# best_model.pth, one periodic checkpoint, and the next best model, written
# before the old one is deleted. That peaks at ~24 GB, so a run pointed at
# /kaggle/working dies of ENOSPC around global step 1760, roughly 45 min in --
# and it takes the notebook with it, because papermill writes
# __notebook__.ipynb to the same volume and a full disk truncates the JSON
# mid-write. Train on /kaggle/temp, which has hundreds of GB, and mirror back
# only the checkpoint that has to survive the session.
DATASET = "/kaggle/temp/female_dataset"     # scratch, not against the 20 GB quota
RUN     = "/kaggle/temp/run"                # checkpoints live here
KEEP    = "/kaggle/working/run"             # resume mirror; config + best model only
EVAL    = "/kaggle/working/eval_out"
BASE    = RUN + "/training/XTTS_v2.0_original_model_files"
os.makedirs("/kaggle/temp", exist_ok=True)


def sh(args, cwd=SRC):
    """Run a step and STOP the notebook if it fails.

    `!cmd` returns quietly on a non-zero exit, so a failed prepare step used to
    let the notebook march on and burn GPU minutes on training that could not
    possibly work. Under Save & Run All that is expensive and the real error
    scrolls far out of sight, so every step that others depend on goes through
    here instead.
    """
    print("$", " ".join(args), flush=True)
    r = subprocess.run(args, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"step failed with exit code {r.returncode}: {' '.join(args)}")


## 3. Inspect the layout before trusting it

The published folders are inconsistent (`Isuru-44100Hz` vs `Yasindu-44100`, and at least
one speaker directory nested inside a duplicate of itself). Look at what is actually there.

In [ ]:
import pathlib, collections
root = pathlib.Path(DATA)
for d in sorted(p for p in root.rglob("*") if p.is_dir()):
    wavs = list(d.glob("*.wav"))
    csvs = list(d.glob("*.csv"))
    if wavs or csvs:
        print(f"{str(d.relative_to(root)):45s} {len(wavs):5d} wav  {[c.name for c in csvs]}")

meta = sorted(root.rglob("metadata.csv"))
print("\nmetadata files:", [str(m.relative_to(root)) for m in meta])
if meta:
    print("\nfirst 3 raw lines of", meta[0].name)
    for line in meta[0].read_text(encoding="utf-8-sig").splitlines()[:3]:
        print("  ", line[:160])

## 4. Build the dataset &mdash; and prove the text tokenises

This **fails loudly** rather than training on garbage if the romanisation contains a
character `sinhala_text.py` does not map, or if any `[UNK]` survives.

In [ ]:
import urllib.request
VOCAB = "/kaggle/temp/vocab.json"
urllib.request.urlretrieve(
    "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json", VOCAB)

sh(["python", "prepare_voicemakers.py",
    "--src", DATA, "--out", DATASET,
    "--speakers", "dinithi", "harini",
    "--vocab", VOCAB, "--eval-per-speaker", "40"])


## 5. Smoke test &mdash; two minutes, catches every wiring fault

In [ ]:
import subprocess, sys
sys.path.insert(0, SRC)
import train_log

# The smoke test exists to catch the one thing a zero exit code does NOT: a loss
# of nan. Training can run for hours writing NaN checkpoints and still exit 0.
# So parse the losses rather than trusting the return code -- see train_log.py,
# which strips the trainer's ANSI colour codes before doing so.
SMOKE_LOG = "/kaggle/working/smoke.log"
cmd = ["python", "train_xtts_female.py", "--dataset", DATASET,
       "--out", "/kaggle/temp/smoke", "--smoke",
       "--batch-size", "2", "--grad-accum", "2"]
print("$", " ".join(cmd), flush=True)
with open(SMOKE_LOG, "w") as fh:
    rc = subprocess.run(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT).returncode

txt = open(SMOKE_LOG, encoding="utf-8", errors="replace").read()
print("\n".join(train_log.interesting(txt, 12)))

if rc != 0:
    print(train_log.strip_ansi(txt)[-3000:])
    raise RuntimeError(f"smoke test exited {rc} -- full output in {SMOKE_LOG}")

values = train_log.losses(txt)
if not values:
    raise RuntimeError("smoke test printed no loss_mel_ce -- training never started")
bad = train_log.nonfinite(values)
if bad:
    raise RuntimeError(
        f"loss_mel_ce is {bad[0]!r} -- this run would produce only NaN checkpoints.\n"
        "  If --mixed-precision was passed, drop it (it is off by default now).\n"
        "  If fp16 was already off, the cause is data: look for silent or\n"
        "  corrupt clips in the corpus.")
print(f"\nOK -- {len(values)} finite loss_mel_ce values, "
      f"first {values[0]} -> last {values[-1]}")


In [ ]:
# The smoke run writes the same ~5.5 GB checkpoints as the real one; drop them
# before training starts rather than carrying them for the next eight hours.
!rm -rf /kaggle/temp/smoke
!df -h /kaggle/working /kaggle/temp | grep -v Filesystem

## 6. Train

Backgrounded so the notebook stays responsive. Effective batch is
`batch_size x grad_accum = 64`. Upstream recommends 252, which is right for a datacentre;
on one T4 that is ~100 s per optimiser step and a whole session buys ~400 steps &mdash; far
too few to move the model onto a new sound inventory. Drop `--batch-size` to 3 or 2 on OOM
and raise `--grad-accum` to keep the product near 64.

In [ ]:
import os, shutil, subprocess, sys, time, json, glob
sys.path.insert(0, SRC)
import train_log

# Free space on the volume holding `path`, in GB. Walks up to the nearest
# existing ancestor: RUN does not exist for the first seconds of a run, and
# disk_usage() on a missing path raises -- which would take out the very guard
# that is supposed to stop a crash.
def free_gb(path):
    path = os.path.abspath(path)
    while not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return shutil.disk_usage(path).free / 1e9

# Stop while there is still room to finish writing a checkpoint pair (~11 GB),
# and while /kaggle/working can still hold the notebook papermill writes at the
# end. Hitting ENOSPC instead loses the checkpoint being written AND the
# notebook, which is how an earlier run lost its whole session.
RUN_FLOOR_GB, WORKING_FLOOR_GB = 12.0, 2.0

# Kaggle sets this to "Batch" under Save & Run All, "Interactive" otherwise.
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive").lower() == "batch"
TRAIN_BUDGET_H = 8.5   # leave room for evaluation inside Kaggle's 12 h session cap

# The OOM ladder. Two runs died at exactly global step 5850 with
#   OutOfMemoryError: Tried to allocate 14.00 MiB ... 1.08 GiB reserved but
#   unallocated
# on a 14.56 GiB T4 -- allocator fragmentation after hours of variable-length
# batches, not a batch that never fitted. train_xtts_female.py now sets
# expandable_segments:True, which is the actual fix; this ladder is the fallback
# for when it is not enough. Each rung halves peak activation memory while
# keeping batch_size x grad_accum near 64, so the effective batch -- and
# therefore the learning dynamics -- barely changes.
LADDER = [(4, 16), (3, 21), (2, 32)]

LOG = "/kaggle/working/train.log"


def read_log():
    try:
        with open(LOG, encoding="utf-8", errors="replace") as fh:
            return fh.read()
    except OSError:
        return ""


def find_resume():
    """Latest run dir we can continue: this session's first, then an attached one."""
    here = glob.glob(RUN + "/training/GPT_XTTS_si_female-*")
    here = [p for p in here if os.path.isfile(p + "/best_model.pth")]
    if here:
        return max(here, key=os.path.getmtime), False
    # /kaggle/input is read-only and the trainer writes into the directory it is
    # handed, so a checkpoint from a previous session is copied out first.
    prev = [p for p in glob.glob("/kaggle/input/**/GPT_XTTS_si_female-*", recursive=True)
            if os.path.isfile(p + "/best_model.pth") and os.path.isfile(p + "/config.json")]
    if not prev:
        return None, False
    src = max(prev, key=os.path.getmtime)
    dst = RUN + "/training/" + os.path.basename(src)
    os.makedirs(dst, exist_ok=True)
    for name in ("config.json", "best_model.pth", "speaker_refs.json"):
        if os.path.isfile(src + "/" + name):
            shutil.copy2(src + "/" + name, dst + "/" + name)
    return dst, True


def launch(batch, accum, resume, deadline):
    """One training process, guarded. Returns (returncode, reason)."""
    cmd = ["python", "train_xtts_female.py",
           "--dataset", DATASET, "--out", RUN,
           "--epochs", "40", "--batch-size", str(batch), "--grad-accum", str(accum),
           "--lr", "1e-5", "--save-step", "1000"]
    if resume:
        cmd += ["--continue-path", resume]
    print("\n$ " + " ".join(cmd), flush=True)

    with open(LOG, "a") as fh:
        proc = subprocess.Popen(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT)

    if not BATCH:
        print("pid", proc.pid, "-- training in the background")
        return None, "backgrounded"

    last_beat, died_of_nan, out_of_disk = 0.0, False, None
    while proc.poll() is None and time.time() < deadline:
        time.sleep(30)
        # Checked every loop, not every heartbeat -- a checkpoint save can fill
        # the volume in well under the ten minutes between reports.
        if free_gb(RUN) < RUN_FLOOR_GB:
            out_of_disk = f"{RUN} is down to {free_gb(RUN):.1f} GB"
        elif free_gb("/kaggle/working") < WORKING_FLOOR_GB:
            out_of_disk = f"/kaggle/working is down to {free_gb('/kaggle/working'):.1f} GB"
        if out_of_disk:
            print(f"\ndisk guard: {out_of_disk} -- stopping now so the checkpoint "
                  "already on disk stays usable and the notebook can still be "
                  "written.", flush=True)
            break
        if time.time() - last_beat > 600:          # heartbeat every ~10 min
            last_beat = time.time()
            txt = read_log()
            left = (deadline - time.time()) / 3600
            print(f"[{left:5.2f} h left] [disk {free_gb(RUN):.0f} GB] "
                  + " | ".join(t[-160:] for t in train_log.interesting(txt, 2)),
                  flush=True)
            # Do not sit through eight hours of NaN. The smoke test should have
            # caught it, but belt and braces.
            if train_log.diverged(train_log.losses(txt)):
                died_of_nan = True
                print("\nloss_mel_ce has been nan for several reports -- aborting. "
                      "Every checkpoint from here would be NaN.", flush=True)
                break

    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()

    # -15 is SIGTERM from our own terminate(), so it is not a crash.
    reason = ("nan" if died_of_nan else "disk" if out_of_disk
              else "budget" if time.time() >= deadline
              else "epochs_done" if proc.returncode == 0 else "process_exit")
    return proc.returncode, reason


# ---------------------------------------------------------------- run ----
open(LOG, "w").close()
resume, from_input = find_resume()
if resume:
    print(("RESUMING from an attached session: " if from_input else "resuming: ") + resume)
else:
    print("fresh run -- no previous GPT_XTTS_si_female-* found")
print("mode:", "BATCH (blocking)" if BATCH else "INTERACTIVE (backgrounded)")

started = time.time()
deadline = started + TRAIN_BUDGET_H * 3600
attempts = []

for rung, (batch, accum) in enumerate(LADDER):
    rc, reason = launch(batch, accum, resume, deadline)
    attempts.append({"batch_size": batch, "grad_accum": accum,
                     "returncode": rc, "reason": reason})
    if reason == "backgrounded":
        break

    tail = [l for l in train_log.strip_ansi(read_log()).splitlines() if l.strip()][-40:]
    oom = any("OutOfMemoryError" in l or "CUDA out of memory" in l for l in tail)
    print(f"\nattempt {rung + 1}: batch {batch} x {accum}, exit {rc}, reason {reason}")

    # Only an OOM is worth retrying smaller, and only if the budget has time
    # left. Anything else -- nan, disk, budget, a clean finish -- means a smaller
    # batch changes nothing, so stop rather than burn the session proving it.
    if not (reason == "process_exit" and oom and time.time() < deadline - 600):
        break
    if rung + 1 >= len(LADDER):
        print("\nOOM again at the smallest rung -- out of ladder.", flush=True)
        break
    resume, _ = find_resume()   # continue from what this attempt reached
    print(f"\nCUDA OOM. Retrying at batch {LADDER[rung+1][0]} x {LADDER[rung+1][1]} "
          f"from {resume}, {(deadline - time.time())/3600:.1f} h of budget left.",
          flush=True)

if BATCH:
    last = attempts[-1]
    tail = [l for l in train_log.strip_ansi(read_log()).splitlines() if l.strip()][-40:]
    with open("/kaggle/working/run_status.json", "w", encoding="utf-8") as fh:
        json.dump({"reason": last["reason"], "returncode": last["returncode"],
                   "wall_h": round((time.time() - started) / 3600, 2),
                   "budget_h": TRAIN_BUDGET_H, "attempts": attempts,
                   "log_tail": tail}, fh, indent=1)
    print("\nstop reason:", last["reason"],
          "| wall", round((time.time() - started) / 3600, 2), "h of", TRAIN_BUDGET_H,
          "|", len(attempts), "attempt(s)")

    if last["reason"] == "process_exit":
        print("\n--- training exited unexpectedly; last lines of train.log ---")
        for _l in tail:
            print(_l)
        print("-" * 78)

    # Mirror what a resume needs onto the persisted volume. --continue-path wants
    # a run directory holding config.json and a checkpoint; best_model.pth carries
    # the optimizer state too, so those two files are the whole resume kit. The
    # periodic checkpoints stay on /kaggle/temp and are discarded with the session.
    runs = glob.glob(RUN + "/training/GPT_XTTS_si_female-*")
    if runs:
        latest = max(runs, key=os.path.getmtime)
        dest = KEEP + "/training/" + os.path.basename(latest)
        os.makedirs(dest, exist_ok=True)
        for name in ("config.json", "best_model.pth", "speaker_refs.json"):
            srcf = latest + "/" + name
            if os.path.isfile(srcf):
                shutil.copy2(srcf, dest + "/" + name)
                print(f"  kept {name}  {os.path.getsize(srcf)/1e9:.1f} GB")
        print("resume next session with --continue-path", dest)

    if last["reason"] == "nan":
        raise RuntimeError(
            "training diverged to nan. fp16 is off by default now; if you enabled "
            "--mixed-precision, remove it. Otherwise check the corpus for silent "
            "or corrupt clips.")


In [ ]:
# Re-run to follow along. loss_mel_ce is the acoustic reconstruction term and the
# only one that tracks audio quality; loss_text_ce carries weight 0.01.
!grep -E "loss_mel_ce|EPOCH|EVAL|BEST" /kaggle/working/train.log | tail -n 25

### Training curves

`loss_mel_ce` is the acoustic reconstruction term and the only loss that tracks audio
quality &mdash; `loss_text_ce` carries weight 0.01. Two curves, answering different
questions: **train** says whether the model is fitting at all, **eval** says whether it
is generalising. Once eval turns up while train keeps falling, every later checkpoint is
worse than one already on disk, and the export should come from `best_model.pth` rather
than the last step. The verdict printed below states which case this run is in.

In [ ]:
# Curves survive in the notebook output; the TensorBoard events do not -- they
# live in the run directory, which Kaggle deletes with the session.
sh(["python", "plot_training.py", "--log", LOG,
    "--out", "/kaggle/working/curves.png",
    "--title", "XTTS-v2 Sinhala female -- Dinithi + Harini"])

from IPython.display import Image, display
display(Image("/kaggle/working/curves.png"))


### Resuming after the 12 h limit
Turn on **Persistence &rarr; Files**. Next session, re-run cells 1&ndash;4 then this instead of
the training cell above.

In [ ]:
# The mirror written by the training cell is what survives the session, so
# resume points at KEEP and trains onward into RUN on /kaggle/temp again.
# import glob, os
# prev = max(glob.glob(KEEP + "/training/GPT_XTTS_si_female-*"), key=os.path.getmtime)
# !cd {SRC} && python train_xtts_female.py --dataset {DATASET} --out {RUN} \
#     --epochs 40 --batch-size 4 --grad-accum 16 --lr 1e-5 --continue-path {prev}

## 7. Objective evaluation

MCD, log-F0 RMSE, F0 correlation, speaker similarity, duration ratio, generation failure
rate and RTF over the held-out split. Add `--utmos` for the learned MOS predictor, and
`--asr openai/whisper-large-v3` for the CER gap (slow, large download).

In [ ]:
import glob, os
runs = glob.glob(RUN + "/training/GPT_XTTS_si_female-*")
if not runs:
    raise RuntimeError("no training run directory found -- training did not produce "
                       "a checkpoint. Check /kaggle/working/train.log.")
run = max(runs, key=os.path.getmtime)
print("run:", run)

sh(["python", "evaluate_xtts.py", "--run", run, "--base", BASE,
    "--dataset", DATASET, "--out", EVAL, "--n", "40", "--utmos"])


In [ ]:
from IPython.display import Audio, Markdown, display
import glob, json

display(Markdown(open(EVAL + "/report.md", encoding="utf-8").read()))

ref = json.load(open(DATASET + "/eval_reference.json", encoding="utf-8"))
for it in ref[:4]:
    syn = EVAL + "/synth/" + it["clip_id"] + ".wav"
    if not glob.glob(syn):
        continue
    print("\n" + it["sinhala"])
    print("  speaker:", it["speaker"])
    print("  REAL recording:");  display(Audio(DATASET + "/" + it["wav"]))
    print("  SYNTHESISED:");     display(Audio(syn))

## 7b. Experiments &mdash; which checkpoint, and which decode settings

Run 4 established two things that only evaluation can settle. The checkpoint with
the best `loss_mel_ce` was **not** the one with the best output &mdash; run 3's model
had 0 % failures at a worse loss. And the only clear regressions, failure rate
0 &rarr; 3.8 % and duration ratio 1.026, are **decoding** behaviour, so they can move
on the weights that already exist.

Each experiment is a full evaluation of 40 clips per speaker, so this belongs in a
session where training is skipped: attach the previous run as an input, run cells
1&ndash;4, then come straight here. Every experiment keeps its own directory and
appends one row to `/kaggle/working/experiments/experiments.csv`. Nothing is
overwritten, and an experiment that already has a `metrics.json` is skipped, so a
sweep that runs out of session continues in the next one.


In [ ]:
# Experiments. Each one is a FULL evaluation of EXP_N clips per speaker -- synthesis
# plus pyin, which is the slow part -- so this is minutes per experiment, not seconds.
# After an 8.5 h training run there is not enough session left for a real sweep, so
# the intended use is a SEPARATE session: attach the previous run as an input, run
# cells 1-4, skip the training cell, and come straight here.
#
# Everything is resumable. An experiment whose metrics.json already exists is
# skipped, so a sweep cut short by the 12 h limit continues where it stopped and
# nothing already measured is recomputed or overwritten.
import os, shutil, subprocess, time, csv

SWEEP    = "/kaggle/temp/sweeps"            # synth wavs: not against the 20 GB quota
KEEPEXP  = "/kaggle/working/experiments"    # register + summaries: must survive
REGISTRY = KEEPEXP + "/experiments.csv"     # one row per experiment, every sweep
os.makedirs(KEEPEXP, exist_ok=True)

EXP_N = 40            # clips per speaker; 40 is what every row in RESULTS.md used
EXP_SEED = 1234       # the baseline's seed, or the comparison is not a comparison

# Step 2 -- which checkpoint synthesises best, as opposed to which has the lowest
# eval loss. Run 4 reached the best loss of any run and came back with 3.8 %
# failures against run 3's 0 %, so those are not the same question.
RUN_CHECKPOINT_SWEEP = True
# Step 3 -- decoding. Failure rate and duration ratio are decode behaviour and cost
# no training. One axis at a time against a fixed baseline: the full product of
# four lists is dozens of evaluations and tells you less about which one moved it.
RUN_DECODE_SWEEP = True
# Step 4 -- one text path for both speakers. This is the INFERENCE half; the
# training half is prepare_voicemakers.py --text-path script, which needs a retrain.
RUN_TEXTPATH_SWEEP = False


def sweep(name, extra, note=""):
    """One sweep into its own directory, sharing the one register."""
    out = f"{SWEEP}/{name}"
    print(f"\n{'#' * 78}\n# {name}   {note}\n{'#' * 78}", flush=True)
    cmd = ["python", "sweep_eval.py", "--run", run, "--base", BASE,
           "--dataset", DATASET, "--out", out, "--registry", REGISTRY,
           "--n", str(EXP_N), "--seed", str(EXP_SEED)] + extra
    print("$ " + " ".join(cmd), flush=True)
    # Deliberately not sh(): a sweep that fails must not throw away the sweeps
    # already recorded, and sweep_eval writes the register after every experiment.
    rc = subprocess.run(cmd, cwd=SRC).returncode
    if rc != 0:
        print(f"!! {name} exited {rc} -- whatever it measured before failing is "
              "still in the register", flush=True)
    summary = out + "/summary.md"
    if os.path.isfile(summary):
        shutil.copy2(summary, f"{KEEPEXP}/{name}_summary.md")
        print(open(summary, encoding="utf-8").read())
    return rc


t0 = time.time()

if RUN_CHECKPOINT_SWEEP:
    sweep("checkpoints", ["--all-checkpoints", "--utmos"],
          "step 2 -- one evaluation per checkpoint")

if RUN_DECODE_SWEEP:
    # Lower temperature and higher repetition_penalty both attack over-generation;
    # swept separately so the result says WHICH one moved the number.
    sweep("decode_temperature",
          ["--checkpoints", run + "/best_model.pth", "--temperature", "0.75,0.65,0.6"],
          "step 3a -- temperature")
    sweep("decode_reppen",
          ["--checkpoints", run + "/best_model.pth",
           "--repetition-penalty", "5.0,7.5,10.0"],
          "step 3b -- repetition_penalty")
    sweep("decode_topk_topp",
          ["--checkpoints", run + "/best_model.pth",
           "--top-k", "50,30", "--top-p", "0.85,0.8"],
          "step 3c -- top_k and top_p")

if RUN_TEXTPATH_SWEEP:
    sweep("textpath",
          ["--checkpoints", run + "/best_model.pth", "--text-from", "dataset,script"],
          "step 4 -- one ascii path for both speakers")

print(f"\nsweeps took {(time.time() - t0) / 3600:.2f} h")
if os.path.isfile(REGISTRY):
    rows = list(csv.DictReader(open(REGISTRY, encoding="utf-8")))
    print(f"\nregister: {REGISTRY}  --  {len(rows)} experiment(s) recorded")
    print("\nDownload experiments.csv. It is the only record that survives the "
          "session, and\na change belongs in it before it goes into RESULTS.md as "
          "an improvement.")
else:
    print("\nNo register written -- every sweep was disabled, or all of them failed.")


## 8. Build the MOS / SUS listening panel

The two metrics the literature actually compares on need human ears. This writes one
self-contained HTML file &mdash; download it from the output pane and send it to native
speakers; they rate in a browser and send back a CSV.

In [ ]:
sh(["python", "listening_test.py", "--run", run, "--base", BASE,
    "--dataset", DATASET, "--out", "/kaggle/working/listening_test"])
print()
for p in sorted(os.listdir("/kaggle/working/listening_test")):
    full = "/kaggle/working/listening_test/" + p
    if os.path.isfile(full):
        print(f"  {p}  {os.path.getsize(full)/1e6:.1f} MB")


## 9. Export the model

`model.pth` + `config.json` + `vocab.json` in one folder. Always run text through
`sinhala_text.to_ascii()` before synthesising &mdash; raw Sinhala gives `[UNK]` and noise.

In [ ]:
import shutil, os, glob
EXP = "/kaggle/working/xtts_si_female"
os.makedirs(EXP, exist_ok=True)
ck = run + "/best_model.pth"
if not os.path.isfile(ck):
    ck = KEEP + "/training/" + os.path.basename(run) + "/best_model.pth"
if not os.path.isfile(ck):
    ck = max(glob.glob(run + "/checkpoint_*.pth"),
             key=lambda p: int(p.split("_")[-1].split(".")[0]))
# Hardlink where possible: on Kaggle the resume mirror and the export are both
# on /kaggle/working, and a copy would spend another 5.5 GB of the 20 GB quota
# on bytes that already exist.
try:
    os.link(ck, EXP + "/model.pth")
except (OSError, AttributeError):
    shutil.copy2(ck, EXP + "/model.pth")
for f in ("config.json", "vocab.json"):
    shutil.copy2(BASE + "/" + f, EXP + "/" + f)
shutil.copy2(CODE + "/xtts_sinhala/sinhala_text.py", EXP + "/sinhala_text.py")
print("exported from", ck)
!du -sh {EXP} && ls -la {EXP}

## 9b. Shrink it &mdash; 5.6 GB &rarr; 1.9 GB &rarr; 0.95 GB

The exported file is a **trainer** checkpoint. Roughly two thirds of it is AdamW
state plus three modules (`dvae` and two mel encoders) that build training targets
and that coqui's own loader discards on the way in. Dropping them is not a
tradeoff: `--verify` checks every surviving tensor for identical dtype, shape and
bytes, so "the weights are untouched" is proved rather than asserted.

`--fp16` is a different claim. It halves the file by rounding every weight, and
`load_state_dict` casts back to fp32 on load &mdash; so the arithmetic is unchanged
and only the stored precision is lost. Small, not zero. The next cell is what
decides whether it ships.


In [ ]:
# Steps 8-10: 5.6 GB trainer checkpoint -> 1.9 GB inference model -> 0.95 GB fp16.
#
# The strip is free and provable: optimizer state, scheduler state, the gradient
# scaler, step/epoch metadata and the three trainer-only modules are all things
# inference never reads, and --verify proves the weights that remain are bit-for-bit
# the originals. fp16 is NOT free -- it rounds every weight -- so it is written here
# and gated in the next cell before anything calls it a deployment candidate.
import os, shutil, subprocess

OPT  = CODE + "/xtts_model_female_optimized"
SLIM = EXP + "/model_slim.pth"      # ~1.9 GB, fp32, identical inference
FP16 = EXP + "/model_fp16.pth"      # ~0.95 GB, weights rounded to fp16

# Both new files land on the 20 GB /kaggle/working beside the 5.5 GB export, and
# torch.save writes the whole tensor set before the old file is released. Refuse
# now rather than half-write a checkpoint at 90 % and lose the notebook with it.
need_gb = 3.5
free_gb_working = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"/kaggle/working: {free_gb_working:.1f} GB free, need ~{need_gb} GB")
if free_gb_working < need_gb:
    raise RuntimeError(
        f"only {free_gb_working:.1f} GB free on /kaggle/working. The slim model is "
        "~1.9 GB and the fp16 model ~0.95 GB. Delete the 5.5 GB export (the resume "
        "mirror in /kaggle/working/run is the copy that matters) or write these to "
        "/kaggle/temp and download them from there.")

# ck is the checkpoint the export cell chose. Never overwritten: optimize_checkpoint
# refuses an --out that already exists, so re-running this cell after a change means
# deleting the old file on purpose.
print("source checkpoint:", ck, f"{os.path.getsize(ck) / 1e9:.2f} GB")

for out_path, extra in ((SLIM, []), (FP16, ["--fp16"])):
    if os.path.isfile(out_path):
        print(f"\n{out_path} exists already -- skipping. Delete it to rebuild.")
        continue
    sh(["python", "optimize_checkpoint.py", "--in", ck, "--out", out_path,
        "--strip", "--hashes"] + extra, cwd=OPT)

# Step 9, first half: the strip is bit-identical, proved without a GPU and without
# synthesising anything. This is a stronger statement than "the metrics matched",
# and it is what makes the exact-equality check in the next cell mean something.
print("\n" + "=" * 78)
print("VERIFYING the stripped fp32 model against the checkpoint it came from")
print("=" * 78)
sh(["python", "optimize_checkpoint.py", "--in", ck, "--out", SLIM, "--verify"], cwd=OPT)

print("\n" + "=" * 78)
print("VERIFYING the fp16 model -- shapes and keys only; rounding is expected")
print("=" * 78)
sh(["python", "optimize_checkpoint.py", "--in", ck, "--out", FP16, "--verify"], cwd=OPT)

for name in ("model.pth", "model_slim.pth", "model_fp16.pth"):
    p = EXP + "/" + name
    if os.path.isfile(p):
        print(f"{name:18} {os.path.getsize(p) / 1e9:6.2f} GB")


## 9c. Prove neither file cost anything &mdash; or fail

Two comparisons against the **existing** evaluation pipeline, both reporting MCD,
F0 RMSE, F0 correlation, SECS, duration ratio, failure rate, UTMOS and per-speaker
metrics:

| | Expected | Meaning of a failure |
|---|---|---|
| slim vs the checkpoint | **exactly** equal | something other than the strip changed |
| fp16 vs slim | within tolerance | rounding cost real quality &mdash; ship the 1.9 GB file instead |

`compare_quality.py` exits non-zero on a regression and `sh()` stops the notebook
on a non-zero exit, so a failed gate cannot scroll past unnoticed.


In [ ]:
# Steps 9 and 11: does either smaller file cost anything measurable?
#
# Two comparisons, and they are not the same kind of claim:
#
#   slim vs the checkpoint   must come back EXACTLY equal. The weights are
#                            bit-identical (the previous cell proved it) and the
#                            evaluation is seeded, so any difference at all would
#                            mean something else moved.
#   fp16 vs slim             weights are rounded, so the sampled token sequence
#                            diverges and every metric moves a little. This is
#                            the comparison that decides whether fp16 ships.
#
# compare_quality.py exits non-zero on a regression beyond tolerance, and sh()
# stops the notebook on a non-zero exit -- so a failing gate is not something that
# scrolls past.
import os

CMP = "/kaggle/temp/cmp"
KEEPCMP = "/kaggle/working/experiments"
os.makedirs(KEEPCMP, exist_ok=True)

# Decoding must match what the sweeps chose, on both sides of both comparisons.
# Leave these at the defaults until a sweep has actually beaten them.
DEC = ["--temperature", "0.75", "--repetition-penalty", "5.0",
       "--top-k", "50", "--top-p", "0.85", "--length-penalty", "1.0"]
GATE_N = 40
SEEDS = "1234"          # widen to "1234,1235,1236" when a result sits on a boundary


def gate(baseline, candidate, name):
    print(f"\n{'#' * 78}\n# {name}\n{'#' * 78}", flush=True)
    sh(["python", "compare_quality.py", "--run", run, "--base", BASE,
        "--dataset", DATASET, "--baseline", baseline, "--candidate", candidate,
        "--out", f"{CMP}/{name}", "--registry", KEEPCMP + "/experiments.csv",
        "--n", str(GATE_N), "--seeds", SEEDS, "--utmos"] + DEC, cwd=OPT)


# Step 9 -- the lossless one. Expect "Metrics are EXACTLY equal".
gate(ck, SLIM, "slim_vs_checkpoint")

# Step 11 -- the one that can fail. Baseline is the SLIM model, not the raw
# checkpoint: fp16 is a change to the slim model, and comparing it against the
# checkpoint would fold two changes into one verdict.
gate(SLIM, FP16, "fp16_vs_slim")

print("\nBoth gates passed. 'Passed' means no metric moved beyond tolerance at "
      "this\nseed and this n -- it is not a claim that the models are identical, "
      "except for\nthe slim comparison, where the exact-equality line says so "
      "explicitly.")


## 9d. What each file costs to run

File size, load time, peak VRAM, RTF and **time to first audio**, one row per
configuration. Latency and RTF are different numbers: a model can start streaming
in 300 ms and still have an unimpressive RTF, and a reader application cares about
the first one.

Nothing here is gated &mdash; speed is expected to change, and the cell above is what
decides whether it is allowed to.


In [ ]:
# Step 12: what each file costs to run -- size, load time, peak VRAM, RTF, and time
# to the first audio chunk. One row per configuration in results.csv.
#
# Nothing here is gated. Speed is expected to change; the previous cell is what
# decides whether a configuration is allowed to. RTF and latency are different
# numbers and both are reported: RTF is compute per second of audio, latency is how
# long a listener waits before anything plays.
import glob, json, os, shutil

RESULTS = "/kaggle/working/experiments/results.csv"
os.makedirs(os.path.dirname(RESULTS), exist_ok=True)

# One speaker reference, the same one for every row -- conditioning audio changes
# how many tokens get generated, so varying it would vary the thing being measured.
refs = run + "/speaker_refs.json"
if os.path.isfile(refs):
    REF = sorted(json.load(open(refs)).values())[0]
else:
    REF = sorted(glob.glob(DATASET + "/wavs/*/*.wav"))[0]
print("reference wav:", REF)

for tag, ckpt, extra in (
        ("exported",     ck,   []),          # the 5.6 GB trainer checkpoint
        ("slim-fp32",    SLIM, []),          # 1.9 GB, identical weights
        ("fp16-storage", FP16, []),          # 0.95 GB, rounded weights, fp32 maths
        # fp16 COMPUTE is a different thing again: same file, arithmetic in half
        # precision. It changes results, so it would need its own quality gate
        # before it could ship -- measured here only to show what it would buy.
        ("fp16-compute", SLIM, ["--half"])):
    if not os.path.isfile(ckpt):
        print(f"skipping {tag}: {ckpt} not found")
        continue
    sh(["python", "benchmark.py", "--checkpoint", ckpt, "--base", BASE,
        "--ref", REF, "--tag", tag, "--out", RESULTS], cwd=OPT)

print("\n" + open(RESULTS, encoding="utf-8").read())
print("Sizes, VRAM and load time are properties of the file. RTF and latency are "
      "properties\nof this GPU -- a T4 here says nothing about the card the model "
      "will be served on.")


## 10. The next-run brief

Kaggle deletes the session shortly after it ends, so everything needed to plan the next
run is assembled now, while the files still exist. Each recommendation is **derived from
this run's numbers** and carries the evidence that produced it &mdash; it is not a
checklist. Download `next_run.md` from the output pane; it is self-contained.

In [ ]:
# Reads prepare_report.json, train.log and eval_out/metrics.json, and writes one
# markdown file. Missing inputs are reported as missing rather than raising: a
# partial brief still beats reconstructing this by hand after the session is gone.
sh(["python", "next_run_report.py",
    "--dataset", DATASET, "--log", LOG, "--eval", EVAL, "--run", run,
    "--out", "/kaggle/working/next_run.md"])

from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/next_run.md", encoding="utf-8").read()))
